# Phase 1 — Foundation & Data

**Goal:** Install dependencies, access `fuxi-robot/excavator-video` via HuggingFace, extract 50 sample frames, verify CLAHE preprocessing.

**Run all cells top to bottom. Do not skip Cell 1.**

In [ ]:
# Cell 1 — Repo sync (ALWAYS run first)
import os, sys

repo_path = "/kaggle/working/trailer-counter"
if not os.path.exists(repo_path):
    os.system("git clone https://github.com/Rutwik1000/trailer-counter.git " + repo_path)
os.chdir(repo_path)
os.system("git fetch origin")
os.system("git reset --hard origin/master")  # always match remote exactly
os.system("git log --oneline -3")

# Ensure src/ is importable
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

In [ ]:
# Cell 2 — Install dependencies
os.system("pip install -r requirements.txt -q")

# Force reimport of src modules
for mod in list(sys.modules.keys()):
    if mod.startswith("src"):
        del sys.modules[mod]

In [ ]:
# Cell 3 — Verify environment
import torch
print("Python:", sys.version)
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

import cv2, numpy as np
from datasets import load_dataset
from huggingface_hub import hf_hub_download, login
from src.preprocessor import apply_clahe
print("All imports OK")

In [ ]:
# Cell 4 — HuggingFace authentication
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret("hf_token")
login(token=token, add_to_git_credential=False)
print("HF login OK")

In [ ]:
# Cell 5 — Download SiteSense model weights
os.makedirs("models", exist_ok=True)

weight_files = [
    "rfdetr_construction.pth",
    "dinov3_reid_head.pth",
    "osnet_x0_25_msmt17.pt",
]

for filename in weight_files:
    dest = os.path.join("models", filename)
    if os.path.exists(dest):
        print("Already exists:", filename)
        continue
    hf_hub_download(
        repo_id="Zaafan/sitesense-weights",
        filename=filename,
        local_dir="models/",
    )
    print("Downloaded:", filename)

for filename in weight_files:
    assert os.path.exists(os.path.join("models", filename)), "Missing: " + filename
print("All model weights present")

In [ ]:
# Cell 6 — Extract 50 sample frames from 5 fuxi-robot videos
# VideoDecoder confirmed: CHW uint8 RGB [0,255], shape [3, 720, 1280]
import glob
from tqdm import tqdm

os.makedirs("data/frames", exist_ok=True)

TARGET_VIDEOS = 5
FRAMES_PER_VIDEO = 10

ds = load_dataset("fuxi-robot/excavator-video", split="train", streaming=True)

frames_saved = 0
video_count = 0

for sample in tqdm(ds, total=TARGET_VIDEOS, desc="Videos"):
    if video_count >= TARGET_VIDEOS:
        break

    video = sample["video"]
    n_frames = len(video)
    step = max(1, n_frames // FRAMES_PER_VIDEO)

    for i in range(FRAMES_PER_VIDEO):
        frame_idx = i * step
        if frame_idx >= n_frames:
            break
        # CHW uint8 RGB -> HWC uint8 BGR
        frame_np = video[frame_idx].data.permute(1, 2, 0).numpy()
        frame_bgr = cv2.cvtColor(frame_np, cv2.COLOR_RGB2BGR)
        out_path = os.path.join("data/frames", "vid{:02d}_frame{:02d}.jpg".format(video_count, i))
        cv2.imwrite(out_path, frame_bgr)
        frames_saved += 1

    video_count += 1

print("Saved", frames_saved, "frames from", video_count, "videos")

frame_files = glob.glob("data/frames/*.jpg")
assert len(frame_files) >= 40, "Expected >=40 frames, got " + str(len(frame_files))
print("Frame extraction OK:", len(frame_files), "frames")

In [ ]:
# Cell 7 — Visual inspection: 3x3 grid of raw frames
import matplotlib.pyplot as plt
from PIL import Image

sample_paths = sorted(glob.glob("data/frames/*.jpg"))[:9]
fig, axes = plt.subplots(3, 3, figsize=(15, 10))
for ax, path in zip(axes.flatten(), sample_paths):
    img = Image.open(path)
    ax.imshow(img)
    ax.set_title(os.path.basename(path), fontsize=8)
    ax.axis("off")
plt.suptitle("Raw frames — confirm cab-mounted downward perspective", fontsize=12)
plt.tight_layout()
plt.show()
print("HUMAN CHECK: excavator arm visible? trailer below? downward view? (yes/no)")

In [ ]:
# Cell 8 — CLAHE verification: side-by-side before/after on 3 frames
check_paths = sorted(glob.glob("data/frames/*.jpg"))[:3]
fig, axes = plt.subplots(3, 2, figsize=(14, 12))

for row, path in enumerate(check_paths):
    raw = cv2.imread(path)
    enhanced = apply_clahe(raw)

    assert enhanced.shape == raw.shape, "Shape mismatch after CLAHE"
    assert enhanced.dtype == raw.dtype, "Dtype mismatch after CLAHE"

    axes[row, 0].imshow(cv2.cvtColor(raw, cv2.COLOR_BGR2RGB))
    axes[row, 0].set_title("Raw: " + os.path.basename(path), fontsize=8)
    axes[row, 0].axis("off")

    axes[row, 1].imshow(cv2.cvtColor(enhanced, cv2.COLOR_BGR2RGB))
    axes[row, 1].set_title("CLAHE enhanced", fontsize=8)
    axes[row, 1].axis("off")

plt.suptitle("CLAHE preprocessing — before vs after", fontsize=12)
plt.tight_layout()
plt.show()
print("CLAHE verification OK")

In [ ]:
# Cell 9 — Update PROGRESS.md and commit back to repo
import datetime, os
from kaggle_secrets import UserSecretsClient

today = datetime.date.today().isoformat()

progress_path = os.path.join(repo_path, "docs/PROGRESS.md")

with open(progress_path, "r") as f:
    content = f.read()

# 1. Update status table row for Phase 1
content = content.replace(
    "| Phase 1 — Foundation & Data | Not started |",
    "| Phase 1 — Foundation & Data | Complete |",
)

# 2. Insert log entry after the first --- separator
progress_entry = (
    "\n## " + today + " — Phase 1 complete\n\n"
    "**Files created:** src/preprocessor.py, tests/test_preprocessor.py, notebooks/01_foundation.ipynb\n"
    "**Tests passing:** 9/9 in tests/test_preprocessor.py (run locally)\n"
    "**Key finding:** fuxi-robot VideoDecoder yields CHW uint8 RGB [0,255] — not float32 [0,1]. Frame conversion updated (no *255).\n"
    "**Blocker:** None\n"
    "**Next:** Phase 2 — Detection Baseline (RF-DETR vs YOLOv8 on sample frames)\n"
)

insert_marker = "---\n"
insert_pos = content.find(insert_marker) + len(insert_marker)
content = content[:insert_pos] + progress_entry + content[insert_pos:]

with open(progress_path, "w") as f:
    f.write(content)

print("PROGRESS.md updated:")
print(progress_entry)

# Configure git credentials (same mechanism as Windows Credential Manager, but ephemeral)
gh_token = UserSecretsClient().get_secret("github_token")
os.system("git config --global credential.helper store")
with open(os.path.expanduser("~/.git-credentials"), "w") as f:
    f.write("https://x-token:" + gh_token + "@github.com\n")

# Commit and push
os.system("git config user.email 'kaggle@trailer-counter'")
os.system("git config user.name 'Kaggle Executor'")
os.system("git add docs/PROGRESS.md")
os.system("git diff --stat HEAD")
os.system("git commit -m 'docs: phase 1 complete — 50 frames extracted, CLAHE verified'")
os.system("git push origin HEAD:master")
print("Pushed to master")